# EDA: Data Klaim Individu × Keuangan Perusahaan — Analisis Actionable

**Fokus analisis** pada hal yang benar-benar berubah dan dapat dimonitor:

| # | Analisis | Pertanyaan Bisnis |
|---|----------|-------------------|
| 1 | **Profil Risiko per Diagnosa (ICD)** | Penyakit apa yang makin mahal/makin banyak? |
| 2 | **Klaim Luar Negeri: Beban Tersembunyi** | Seberapa besar dampak klaim Singapore/Malaysia? |
| 3 | **Efisiensi Proses vs Liabilitas Neraca** | Proses lambat → utang klaim menumpuk? |
| 4 | **Kecukupan Cadangan vs Realisasi** | Cadangan tumbuh seiring beban klaim riil? |

> Data keuangan (STATISTIK) adalah **neraca perusahaan yang sama** — bukan data industri.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 9.5
sns.set_style('whitegrid')

C = {
    'klaim':     '#2196F3',
    'nominal':   '#FF5722',
    'cadangan':  '#9C27B0',
    'investasi': '#4CAF50',
    'reins':     '#FF9800',
    'likuid':    '#F44336',
    'kas':       '#009688',
    'ln':        '#E91E63',
    'dom':       '#3F51B5',
    'warn':      '#FF9800',
    'neutral':   '#9E9E9E',
    'tail':      '#D32F2F',
}

ICD_COLORS = [
    '#E53935','#D81B60','#8E24AA','#5E35B1','#3949AB','#1E88E5',
    '#039BE5','#00ACC1','#00897B','#43A047','#7CB342','#C0CA33',
    '#FDD835','#FFB300','#FB8C00','#F4511E','#6D4C41','#546E7A'
]
print('Libraries loaded.')

## Load & Persiapan Data

In [ ]:
# ── Data Klaim Individu ─────────────────────────────────────────────
df_klaim = pd.read_csv('dataset/Data_Klaim_Enriched.csv',
                       parse_dates=['Tanggal Pembayaran Klaim',
                                    'Tanggal Pasien Masuk RS',
                                    'Tanggal Pasien Keluar RS'])

df_klaim['bulan'] = df_klaim['Tanggal Pasien Masuk RS'].dt.to_period('M').dt.to_timestamp()
df_klaim['is_ln'] = df_klaim['Lokasi RS'].isin(['Singapore','Malaysia','Thailand',
                                                 'Taiwan','Hong Kong','Japan',
                                                 'Tiongkok','Australia','Overseas','Others'])

# ── Statistik Keuangan (wide → long → pivot) ────────────────────────
df_stat_wide = pd.read_csv('dataset/STATISTIK_ASURANSI_merged_2024-2025.csv')

MONTH_MAP = {
    'Januari':'01','Februari':'02','Maret':'03','April':'04',
    'Mei':'05','Juni':'06','Juli':'07','Agustus':'08',
    'September':'09','Oktober':'10','November':'11','Desember':'12'
}

month_cols = [c for c in df_stat_wide.columns if '-20' in str(c)]

def col_to_period(col):
    m, y = col.split('-')
    return f"{y}-{MONTH_MAP[m]}"

df_stat_long = df_stat_wide.melt(
    id_vars=['No','Akun','Account_EN'],
    value_vars=month_cols,
    var_name='bulan_raw', value_name='nilai'
)
df_stat_long['bulan'] = pd.to_datetime(
    df_stat_long['bulan_raw'].apply(col_to_period)
)
df_stat_long = df_stat_long.drop(columns='bulan_raw')

stat_pivot = df_stat_long.pivot_table(
    index='bulan', columns='Akun', values='nilai', aggfunc='first'
).reset_index()

# ── Agregasi klaim bulanan ──────────────────────────────────────────
klaim_bulanan = df_klaim.groupby('bulan').agg(
    n_klaim          = ('Claim ID', 'count'),
    total_nominal    = ('Nominal Klaim Yang Disetujui', 'sum'),
    avg_nominal      = ('Nominal Klaim Yang Disetujui', 'mean'),
    median_nominal   = ('Nominal Klaim Yang Disetujui', 'median'),
    q95_nominal      = ('Nominal Klaim Yang Disetujui', lambda x: x.quantile(0.95)),
    avg_proc_days    = ('Processing_Days', 'mean'),
    total_biaya_rs   = ('Nominal Biaya RS Yang Terjadi', 'sum'),
    avg_selisih      = ('Selisih_Klaim', 'mean'),
    avg_rasio_cov    = ('Rasio_Coverage', 'mean'),
    n_ln             = ('is_ln', 'sum'),
    nom_ln           = ('Nominal Klaim Yang Disetujui', lambda x: x[df_klaim.loc[x.index,'is_ln']].sum()),
).reset_index()

klaim_bulanan['pct_ln'] = klaim_bulanan['n_ln'] / klaim_bulanan['n_klaim'] * 100

# ── Merge ─────────────────────────────────────────────────────────────
df = klaim_bulanan.merge(stat_pivot, on='bulan', how='inner').sort_values('bulan').reset_index(drop=True)

print(f'Dataset gabungan: {df.shape[0]} bulan × {df.shape[1]} kolom')
print(f'Rentang: {df["bulan"].min().strftime("%b-%Y")} → {df["bulan"].max().strftime("%b-%Y")}')
print(f'Total klaim: {df["n_klaim"].sum():,}')
print(f'Total nominal dibayar: Rp {df["total_nominal"].sum()/1e9:.1f} Miliar')

---
## Analisis 1 — Profil Risiko Portofolio Klaim per Diagnosa (ICD)
### Penyakit apa yang makin banyak? Mana yang makin mahal?

In [ ]:
# ── Agregasi per ICD Group × Periode ───────────────────────────────
def label_periode(dt):
    if dt < pd.Timestamp('2024-07-01'):
        return 'H1-2024'
    elif dt < pd.Timestamp('2025-01-01'):
        return 'H2-2024'
    else:
        return 'H1-2025'

df_klaim['periode'] = df_klaim['bulan'].apply(label_periode)

icd_periode = df_klaim.groupby(['ICD_Group','periode']).agg(
    n     = ('Claim ID', 'count'),
    total = ('Nominal Klaim Yang Disetujui', 'sum'),
    avg   = ('Nominal Klaim Yang Disetujui', 'mean'),
).reset_index()

# Top 8 ICD groups by total nominal
top8 = df_klaim.groupby('ICD_Group')['Nominal Klaim Yang Disetujui'].sum().nlargest(8).index.tolist()

# Overall ICD summary
icd_summary = df_klaim.groupby('ICD_Group').agg(
    n_total    = ('Claim ID', 'count'),
    total_nom  = ('Nominal Klaim Yang Disetujui', 'sum'),
    avg_nom    = ('Nominal Klaim Yang Disetujui', 'mean'),
).reset_index()
icd_summary['beban_total'] = icd_summary['total_nom']
icd_summary = icd_summary.sort_values('beban_total', ascending=False)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(17, 12))
fig.suptitle('ANALISIS 1 — Profil Risiko Portofolio Klaim per Diagnosa (ICD)',
             fontsize=14, fontweight='bold')

# ── 1A: Total Beban per ICD Group (horizontal bar) ─────────────────
ax1a = axes[0, 0]
top10 = icd_summary.head(10)
colors_1a = [ICD_COLORS[i % len(ICD_COLORS)] for i in range(len(top10))]
bars1a = ax1a.barh(range(len(top10)), top10['total_nom']/1e9, color=colors_1a, alpha=0.85)
ax1a.set_yticks(range(len(top10)))
ax1a.set_yticklabels(top10['ICD_Group'], fontsize=8)
ax1a.set_xlabel('Total Nominal Klaim (Rp Miliar)')
ax1a.set_title('Top 10 ICD Group by Total Beban', fontweight='bold')
for bar, v in zip(bars1a, top10['total_nom']):
    ax1a.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
              f'Rp{v/1e9:.0f}M', va='center', fontsize=7.5)
ax1a.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:.0f}M'))

# ── 1B: Scatter — Volume vs Avg Nominal (kuadran risiko) ───────────
ax1b = axes[0, 1]
for i, row in icd_summary.iterrows():
    color = ICD_COLORS[list(icd_summary.index).index(i) % len(ICD_COLORS)]
    ax1b.scatter(row['n_total'], row['avg_nom']/1e6, s=row['total_nom']/1e9*3,
                 color=color, alpha=0.7, edgecolors='white', linewidths=0.8)
    ax1b.annotate(row['ICD_Group'][:16],
                  (row['n_total'], row['avg_nom']/1e6),
                  fontsize=6.5, xytext=(4,2), textcoords='offset points')

med_n   = icd_summary['n_total'].median()
med_avg = icd_summary['avg_nom'].median() / 1e6
ax1b.axvline(med_n,   color='gray', lw=1, ls='--', alpha=0.6)
ax1b.axhline(med_avg, color='gray', lw=1, ls='--', alpha=0.6)
ax1b.text(icd_summary['n_total'].max()*0.75, icd_summary['avg_nom'].max()/1e6*0.92,
          '⚠ HIGH RISK\n(banyak & mahal)', fontsize=8, color='red',
          bbox=dict(fc='lightyellow', ec='red', alpha=0.7))
ax1b.set_xlabel('Volume Klaim (jumlah)')
ax1b.set_ylabel('Rata-rata Nominal per Klaim (Juta Rp)')
ax1b.set_title('Peta Risiko: Volume vs Biaya Rata-rata\n(ukuran titik = total beban)', fontweight='bold')
ax1b.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'Rp{v:.0f}Jt'))

# ── 1C: Cost Inflation — avg nominal per ICD group antar periode ────
ax1c = axes[1, 0]
periods = ['H1-2024', 'H2-2024', 'H1-2025']
top5 = icd_summary.head(5)['ICD_Group'].tolist()
x_idx = np.arange(len(periods))
width = 0.15
for i, grp in enumerate(top5):
    sub = icd_periode[icd_periode['ICD_Group'] == grp].set_index('periode')
    vals = [sub.loc[p, 'avg']/1e6 if p in sub.index else 0 for p in periods]
    ax1c.bar(x_idx + i*width, vals, width=width,
             label=grp[:18], color=ICD_COLORS[i], alpha=0.85)
ax1c.set_xticks(x_idx + width*2)
ax1c.set_xticklabels(periods)
ax1c.set_ylabel('Rata-rata Nominal per Klaim (Juta Rp)')
ax1c.set_title('Cost Inflation per Diagnosa — Top 5 ICD\n(Rata-rata nominal per klaim, antar periode)', fontweight='bold')
ax1c.legend(fontsize=7, ncol=1, loc='upper left')
ax1c.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'Rp{v:.0f}Jt'))

# ── 1D: Tren bulanan — Top 5 ICD group, total nominal ──────────────
ax1d = axes[1, 1]
icd_bulanan = df_klaim.groupby(['bulan','ICD_Group'])['Nominal Klaim Yang Disetujui'].sum().reset_index()
for i, grp in enumerate(top5):
    sub = icd_bulanan[icd_bulanan['ICD_Group'] == grp].sort_values('bulan')
    ax1d.plot(range(len(sub)), sub['Nominal Klaim Yang Disetujui']/1e9,
              color=ICD_COLORS[i], lw=2, marker='o', ms=4, label=grp[:20])

# x-labels dari bulan unik
all_bulan = sorted(df_klaim['bulan'].unique())
ax1d.set_xticks(range(len(all_bulan)))
ax1d.set_xticklabels([b.strftime('%b\n%y') for b in all_bulan], fontsize=7)
ax1d.set_ylabel('Total Nominal Klaim (Rp Miliar)')
ax1d.set_title('Tren Bulanan Beban Klaim — Top 5 ICD Group', fontweight='bold')
ax1d.legend(fontsize=7, loc='upper right')
ax1d.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:.0f}M'))

plt.tight_layout()
plt.savefig('charts/analisis1_profil_risiko_icd.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Insight Analisis 1 ──────────────────────────────────────────────
print('=== INSIGHT: PROFIL RISIKO ICD ===')
print(f'\nTop 5 ICD Group by Total Beban:')
for _, row in icd_summary.head(5).iterrows():
    print(f'  {row["ICD_Group"]:<30} | {row["n_total"]:>5} klaim | '
          f'Rp{row["total_nom"]/1e9:.1f}M total | Rp{row["avg_nom"]/1e6:.1f}Jt avg')

# Cek inflasi biaya: H1-2024 → H1-2025
print(f'\nCost Inflation (avg nominal klaim): H1-2024 → H1-2025')
for grp in top5:
    sub = icd_periode[icd_periode['ICD_Group'] == grp].set_index('periode')
    if 'H1-2024' in sub.index and 'H1-2025' in sub.index:
        change = (sub.loc['H1-2025','avg'] - sub.loc['H1-2024','avg']) / sub.loc['H1-2024','avg'] * 100
        arrow = '↑' if change > 0 else '↓'
        print(f'  {grp:<30}: {arrow} {change:+.1f}%')

---
## Analisis 2 — Klaim Luar Negeri: Beban Tersembunyi
### Singapore/Malaysia claims = outlier biaya — seberapa besar dampaknya?

In [ ]:
# ── Persiapan data LN ───────────────────────────────────────────────
df_klaim['lokasi_group'] = df_klaim['Lokasi RS'].apply(
    lambda x: x if x in ['Indonesia','Singapore','Malaysia'] else 'Other Overseas'
)

# Cost multiplier: per lokasi
cost_by_loc = df_klaim.groupby('lokasi_group').agg(
    n     = ('Claim ID','count'),
    avg   = ('Nominal Klaim Yang Disetujui','mean'),
    total = ('Nominal Klaim Yang Disetujui','sum'),
).reset_index()
dom_avg = cost_by_loc.loc[cost_by_loc['lokasi_group']=='Indonesia','avg'].values[0]
cost_by_loc['multiplier'] = cost_by_loc['avg'] / dom_avg

# Cost multiplier per lokasi × ICD group (top 5 ICD)
cost_icd_loc = df_klaim[df_klaim['ICD_Group'].isin(top5)].groupby(
    ['lokasi_group','ICD_Group']
)['Nominal Klaim Yang Disetujui'].mean().reset_index()

# Bulanan LN breakdown
ln_bulanan = df_klaim.groupby(['bulan','lokasi_group']).agg(
    n     = ('Claim ID','count'),
    total = ('Nominal Klaim Yang Disetujui','sum'),
).reset_index()

print('Cost multiplier per lokasi (vs Indonesia=1.0x):')
print(cost_by_loc[['lokasi_group','n','avg','multiplier']].round(2).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(17, 11))
fig.suptitle('ANALISIS 2 — Klaim Luar Negeri: Beban Tersembunyi',
             fontsize=14, fontweight='bold')

lokasi_colors = {'Indonesia': C['dom'], 'Singapore': C['ln'],
                 'Malaysia': C['warn'], 'Other Overseas': C['neutral']}

# ── 2A: Tren Bulanan Nominal per Lokasi (stacked bar) ───────────────
ax2a = axes[0, 0]
all_b = sorted(df_klaim['bulan'].unique())
x = range(len(all_b))
xlabels = [b.strftime('%b\n%y') for b in all_b]

bottom = np.zeros(len(all_b))
for loc in ['Indonesia','Malaysia','Singapore','Other Overseas']:
    sub = ln_bulanan[ln_bulanan['lokasi_group']==loc]
    vals = [sub.loc[sub['bulan']==b,'total'].sum()/1e9 if b in sub['bulan'].values else 0
            for b in all_b]
    ax2a.bar(x, vals, bottom=bottom, label=loc,
             color=lokasi_colors.get(loc,'gray'), alpha=0.85, width=0.7)
    bottom += np.array(vals)

ax2a.set_title('Tren Bulanan Total Nominal per Lokasi RS', fontweight='bold')
ax2a.set_ylabel('Total Nominal Klaim (Rp Miliar)')
ax2a.set_xticks(x); ax2a.set_xticklabels(xlabels, fontsize=7)
ax2a.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:.0f}M'))
ax2a.legend(fontsize=8)

# ── 2B: Distribusi Nominal per Lokasi (box plot) ────────────────────
ax2b = axes[0, 1]
loc_order = ['Indonesia','Malaysia','Singapore','Other Overseas']
data_box = [df_klaim[df_klaim['lokasi_group']==l]['Nominal Klaim Yang Disetujui'].values/1e6
            for l in loc_order]
bp = ax2b.boxplot(data_box, labels=loc_order, patch_artist=True,
                  showfliers=True, flierprops=dict(marker='o', ms=2, alpha=0.3))
for patch, loc in zip(bp['boxes'], loc_order):
    patch.set_facecolor(lokasi_colors.get(loc,'gray'))
    patch.set_alpha(0.7)
ax2b.set_ylabel('Nominal Klaim (Juta Rp)')
ax2b.set_title('Distribusi Nominal Klaim per Lokasi RS\n(outlier = titik di luar box)', fontweight='bold')
ax2b.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'Rp{v:.0f}Jt'))
# Annotate multiplier
for i, row in cost_by_loc.iterrows():
    loc = row['lokasi_group']
    if loc in loc_order:
        xi = loc_order.index(loc) + 1
        ax2b.text(xi, ax2b.get_ylim()[1]*0.92, f'{row["multiplier"]:.1f}x',
                  ha='center', fontsize=9, fontweight='bold',
                  color=lokasi_colors.get(loc,'gray'))
ax2b.text(0.99, 0.99, 'Angka = cost multiplier\nvs Indonesia (1.0x)',
          transform=ax2b.transAxes, ha='right', va='top', fontsize=7.5,
          bbox=dict(boxstyle='round', fc='lightyellow', ec='gray'))

# ── 2C: Cost Multiplier per ICD Group ────────────────────────────────
ax2c = axes[1, 0]
pivot_icd_loc = cost_icd_loc.pivot(index='ICD_Group', columns='lokasi_group',
                                    values='Nominal Klaim Yang Disetujui')
if 'Indonesia' in pivot_icd_loc.columns:
    for col in ['Singapore','Malaysia','Other Overseas']:
        if col in pivot_icd_loc.columns:
            pivot_icd_loc[f'mult_{col}'] = pivot_icd_loc[col] / pivot_icd_loc['Indonesia']

mult_cols = [c for c in pivot_icd_loc.columns if c.startswith('mult_')]
if mult_cols:
    pivot_icd_loc_plot = pivot_icd_loc[mult_cols].dropna(how='all')
    pivot_icd_loc_plot.columns = [c.replace('mult_','') for c in pivot_icd_loc_plot.columns]
    pivot_icd_loc_plot.plot(kind='bar', ax=ax2c, width=0.7, alpha=0.85,
                             color=[C['ln'],C['warn'],C['neutral']][:len(pivot_icd_loc_plot.columns)])
    ax2c.axhline(1.0, color='black', lw=1.2, ls='--', label='1.0x = setara domestik')
    ax2c.set_xlabel('')
    ax2c.set_xticklabels(ax2c.get_xticklabels(), rotation=30, ha='right', fontsize=7)
    ax2c.set_ylabel('Cost Multiplier vs Indonesia')
    ax2c.set_title('Cost Multiplier per Diagnosa (ICD)\n(berapa kali lipat lebih mahal vs domestik)',
                   fontweight='bold')
    ax2c.legend(fontsize=8)

# ── 2D: Korelasi klaim LN vs Tagihan Klaim Reasuransi ────────────────
ax2d = axes[1, 1]
if 'Tagihan Klaim Reasuransi' in df.columns:
    ax2d_r = ax2d.twinx()
    x2 = range(len(df))
    xlabels2 = [b.strftime('%b\n%y') for b in df['bulan']]

    ax2d.bar(x2, df['nom_ln']/1e9, color=C['ln'], alpha=0.55, width=0.7,
             label='Nominal Klaim LN (Rp Miliar)')
    ax2d_r.plot(x2, df['Tagihan Klaim Reasuransi']/1e6, color=C['reins'],
                lw=2, marker='s', ms=5, label='Tagihan Klaim Reasuransi (kanan, T Rp)')
    ax2d_r.fill_between(x2, df['Tagihan Klaim Reasuransi']/1e6, alpha=0.1, color=C['reins'])

    corr_val = df['nom_ln'].corr(df['Tagihan Klaim Reasuransi'])
    corr_lag1 = df['nom_ln'].corr(df['Tagihan Klaim Reasuransi'].shift(-1))

    ax2d.set_title(f'Klaim LN vs Tagihan Klaim Reasuransi\n'
                   f'(Korelasi bulan sama: {corr_val:.3f} | Lag+1: {corr_lag1:.3f})',
                   fontweight='bold')
    ax2d.set_ylabel('Nominal Klaim LN (Rp Miliar)', color=C['ln'])
    ax2d_r.set_ylabel('Tagihan Klaim Reasuransi (T Rp)', color=C['reins'])
    ax2d_r.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:.0f}T'))
    ax2d.set_xticks(x2); ax2d.set_xticklabels(xlabels2, fontsize=7)
    lines1, lab1 = ax2d.get_legend_handles_labels()
    lines2, lab2 = ax2d_r.get_legend_handles_labels()
    ax2d.legend(lines1+lines2, lab1+lab2, fontsize=7)

plt.tight_layout()
plt.savefig('charts/analisis2_klaim_ln.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Insight Analisis 2 ──────────────────────────────────────────────
print('=== INSIGHT: KLAIM LUAR NEGERI ===')
total_all  = df_klaim['Nominal Klaim Yang Disetujui'].sum()
total_ln   = df_klaim[df_klaim['is_ln']]['Nominal Klaim Yang Disetujui'].sum()
n_ln       = df_klaim['is_ln'].sum()
n_all      = len(df_klaim)

print(f'\nKlaim luar negeri: {n_ln:,} dari {n_all:,} klaim ({n_ln/n_all*100:.1f}%)')
print(f'Kontribusi nominal: Rp{total_ln/1e9:.1f}M dari Rp{total_all/1e9:.1f}M ({total_ln/total_all*100:.1f}%)')
print(f'\nCost multiplier vs Indonesia:')
for _, row in cost_by_loc.iterrows():
    print(f'  {row["lokasi_group"]:<18}: {row["multiplier"]:.2f}x '
          f'(avg Rp{row["avg"]/1e6:.0f}Jt per klaim)')

if 'Tagihan Klaim Reasuransi' in df.columns:
    corr_val = df['nom_ln'].corr(df['Tagihan Klaim Reasuransi'])
    print(f'\nKorelasi klaim LN vs Tagihan Klaim Reasuransi: {corr_val:.3f}')
    if abs(corr_val) < 0.3:
        print('→ Korelasi lemah: klaim LN belum ter-pass sepenuhnya ke reasuradur di bulan yang sama')
    else:
        print('→ Klaim LN berkorelasi dengan tagihan reasuransi')

---
## Analisis 3 — Efisiensi Proses & Outstanding Liability
### Proses klaim yang lambat → menumpuk di Utang Klaim neraca?

In [ ]:
# ── Breakdown Processing Days ───────────────────────────────────────
proc_by_type = df_klaim.groupby(['bulan','Reimburse/Cashless'])['Processing_Days'].mean().reset_index()
proc_by_ln   = df_klaim.groupby(['bulan','is_ln'])['Processing_Days'].mean().reset_index()

# Outlier klaim: processing > 200 hari
outlier = df_klaim[df_klaim['Processing_Days'] > 200].copy()
outlier_icd = outlier.groupby('ICD_Group').agg(
    n   = ('Claim ID','count'),
    avg = ('Processing_Days','mean'),
    nom = ('Nominal Klaim Yang Disetujui','sum')
).reset_index().sort_values('n', ascending=False)

# Korelasi proc_days vs Utang Klaim
if 'Utang Klaim' in df.columns:
    corr_proc_utang = df[['avg_proc_days','Utang Klaim']].dropna().corr().iloc[0,1]
    print(f'Korelasi avg_proc_days vs Utang Klaim: {corr_proc_utang:.4f}')

# Tren Selisih Klaim (biaya RS - klaim disetujui)
print(f'\nOutlier (processing > 200 hari): {len(outlier)} klaim '
      f'({len(outlier)/len(df_klaim)*100:.1f}% dari total)')
print(outlier_icd.head(5).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(17, 11))
fig.suptitle('ANALISIS 3 — Efisiensi Proses & Outstanding Liability',
             fontsize=14, fontweight='bold')

x   = range(len(df))
xbl = [b.strftime('%b\n%y') for b in df['bulan']]

# ── 3A: Tren Processing Days — Reimburse vs Cashless ───────────────
ax3a = axes[0, 0]
for rt, color, style in [('R', C['likuid'], '-'), ('C', C['investasi'], '--')]:
    sub = proc_by_type[proc_by_type['Reimburse/Cashless']==rt].sort_values('bulan')
    label = 'Reimburse' if rt == 'R' else 'Cashless'
    ax3a.plot(range(len(sub)), sub['Processing_Days'], color=color,
              lw=2, ls=style, marker='o', ms=5, label=label)
mean_proc = df_klaim['Processing_Days'].mean()
ax3a.axhline(mean_proc, color='gray', lw=1, ls=':', label=f'Overall avg ({mean_proc:.0f} hari)')
ax3a.set_xticks(range(len(df)))
ax3a.set_xticklabels(xbl, fontsize=7)
ax3a.set_ylabel('Rata-rata Processing Days')
ax3a.set_title('Processing Days: Reimburse vs Cashless', fontweight='bold')
ax3a.legend(fontsize=8)

# ── 3B: Processing Days — LN vs Domestik ────────────────────────────
ax3b = axes[0, 1]
for is_ln, color, label in [(False, C['dom'], 'Domestik'), (True, C['ln'], 'Luar Negeri')]:
    sub = proc_by_ln[proc_by_ln['is_ln']==is_ln].sort_values('bulan')
    ax3b.plot(range(len(sub)), sub['Processing_Days'], color=color,
              lw=2, marker='s', ms=5, label=label)
    ax3b.fill_between(range(len(sub)), sub['Processing_Days'], alpha=0.1, color=color)
ax3b.set_xticks(range(len(df)))
ax3b.set_xticklabels(xbl, fontsize=7)
ax3b.set_ylabel('Rata-rata Processing Days')
ax3b.set_title('Processing Days: Domestik vs Luar Negeri', fontweight='bold')
ax3b.legend(fontsize=8)

# ── 3C: Scatter — avg_proc_days vs Utang Klaim ──────────────────────
ax3c = axes[1, 0]
if 'Utang Klaim' in df.columns:
    valid = df[['avg_proc_days','Utang Klaim','bulan']].dropna()
    sc = ax3c.scatter(valid['avg_proc_days'], valid['Utang Klaim']/1e6,
                      c=range(len(valid)), cmap='plasma', s=80,
                      edgecolors='white', linewidths=0.8, zorder=3)
    if len(valid) > 2:
        z = np.polyfit(valid['avg_proc_days'], valid['Utang Klaim']/1e6, 1)
        p = np.poly1d(z)
        xf = np.linspace(valid['avg_proc_days'].min(), valid['avg_proc_days'].max(), 50)
        ax3c.plot(xf, p(xf), 'r--', lw=1.8, alpha=0.7,
                  label=f'Trend (slope={z[0]:.2f}T/hari)')
    for _, row in valid.iterrows():
        ax3c.annotate(row['bulan'].strftime('%b%y'),
                      (row['avg_proc_days'], row['Utang Klaim']/1e6),
                      fontsize=6.5, xytext=(3,3), textcoords='offset points')
    plt.colorbar(sc, ax=ax3c, label='Waktu (awal→akhir)')
    ax3c.set_xlabel('Avg Processing Days (klaim individu)')
    ax3c.set_ylabel('Utang Klaim Neraca (Triliun Rp)')
    ax3c.set_title(f'Korelasi: Processing Days vs Utang Klaim Neraca\n'
                   f'(r = {corr_proc_utang:.3f})', fontweight='bold')
    ax3c.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:.1f}T'))
    ax3c.legend(fontsize=8)

# ── 3D: Tren Selisih Klaim (coverage gap) ───────────────────────────
ax3d = axes[1, 1]
ax3d_r = ax3d.twinx()

# Selisih = biaya RS - klaim disetujui (negatif = klaim > biaya?)
selisih_monthly = df_klaim.groupby('bulan').agg(
    avg_selisih    = ('Selisih_Klaim', 'mean'),
    avg_rasio_cov  = ('Rasio_Coverage', 'mean'),
).reset_index().sort_values('bulan')

x3d = range(len(selisih_monthly))
xbl3d = [b.strftime('%b\n%y') for b in selisih_monthly['bulan']]

col_selisih = [C['likuid'] if v < 0 else C['investasi']
               for v in selisih_monthly['avg_selisih']]
ax3d.bar(x3d, selisih_monthly['avg_selisih']/1e6, color=col_selisih, alpha=0.75,
          width=0.7, label='Avg Selisih Klaim (Juta Rp)')
ax3d.axhline(0, color='black', lw=1)

ax3d_r.plot(x3d, selisih_monthly['avg_rasio_cov'], color=C['klaim'], lw=2,
             marker='D', ms=5, label='Avg Rasio Coverage (kanan)')
ax3d_r.axhline(1.0, color='red', lw=1, ls='--', alpha=0.6)

ax3d.set_title('Selisih Klaim & Rasio Coverage\n'
               '(Batang merah = klaim > biaya RS, coverage < 100%)', fontweight='bold')
ax3d.set_ylabel('Avg Selisih (Juta Rp)')
ax3d_r.set_ylabel('Avg Rasio Coverage', color=C['klaim'])
ax3d.set_xticks(x3d); ax3d.set_xticklabels(xbl3d, fontsize=7)
ax3d.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:.1f}Jt'))
lines1, lab1 = ax3d.get_legend_handles_labels()
lines2, lab2 = ax3d_r.get_legend_handles_labels()
ax3d.legend(lines1+lines2, lab1+lab2, fontsize=7)

plt.tight_layout()
plt.savefig('charts/analisis3_efisiensi_proses.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Insight Analisis 3 ──────────────────────────────────────────────
print('=== INSIGHT: EFISIENSI PROSES ===')
print(f'\nOverall avg processing days: {df_klaim["Processing_Days"].mean():.1f} hari')

reimb_avg = df_klaim[df_klaim['Reimburse/Cashless']=='R']['Processing_Days'].mean()
cash_avg  = df_klaim[df_klaim['Reimburse/Cashless']=='C']['Processing_Days'].mean()
print(f'Reimburse: {reimb_avg:.1f} hari | Cashless: {cash_avg:.1f} hari '
      f'(Reimburse {reimb_avg-cash_avg:.1f} hari lebih lambat)')

ln_avg  = df_klaim[df_klaim['is_ln']]['Processing_Days'].mean()
dom_avg_p = df_klaim[~df_klaim['is_ln']]['Processing_Days'].mean()
print(f'LN: {ln_avg:.1f} hari | Domestik: {dom_avg_p:.1f} hari '
      f'(LN {ln_avg-dom_avg_p:.1f} hari lebih lambat)')

print(f'\nOutlier (> 200 hari): {len(outlier)} klaim')
print(f'Top ICD group outlier:')
print(outlier_icd.head(3)[['ICD_Group','n','avg']].to_string(index=False))

if 'Utang Klaim' in df.columns:
    print(f'\nKorelasi Processing Days vs Utang Klaim: {corr_proc_utang:.4f}')
    if corr_proc_utang > 0.3:
        print('→ Terdapat pola positif: proses lebih lambat → Utang Klaim di neraca lebih besar')
    else:
        print('→ Korelasi lemah: faktor lain lebih dominan dalam menentukan Utang Klaim')

---
## Analisis 4 — Kecukupan Cadangan vs Realisasi Pembayaran
### Cadangan tumbuh seiring beban klaim riil? Tail risk (klaim sangat besar) meningkat?

In [ ]:
# ── Metrik kecukupan cadangan ───────────────────────────────────────
if 'Cadangan Klaim' in df.columns:
    # Implied reserve ratio: berapa bulan beban yang di-cover
    df['implied_reserve_months'] = df['Cadangan Klaim'] / df['total_nominal'].replace(0, np.nan)

    # Cadangan premi vs rasio coverage
    if 'Cadangan Premi' in df.columns:
        df['cad_premi_per_klaim'] = df['Cadangan Premi'] / df['n_klaim']

# Kuartil distribusi nominal bulanan
quantile_monthly = df_klaim.groupby('bulan')['Nominal Klaim Yang Disetujui'].agg(
    q25   = lambda x: x.quantile(0.25),
    q50   = 'median',
    q75   = lambda x: x.quantile(0.75),
    q95   = lambda x: x.quantile(0.95),
    q99   = lambda x: x.quantile(0.99),
).reset_index().sort_values('bulan')

print('Implied Reserve Months (Cadangan Klaim / Total Nominal Bulan Ini):')
if 'implied_reserve_months' in df.columns:
    print(df[['bulan','implied_reserve_months']].assign(
        bulan=lambda d: d['bulan'].dt.strftime('%b-%Y'),
        implied_reserve_months=lambda d: d['implied_reserve_months'].round(1)
    ).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(17, 11))
fig.suptitle('ANALISIS 4 — Kecukupan Cadangan vs Realisasi Pembayaran',
             fontsize=14, fontweight='bold')

x   = range(len(df))
xbl = [b.strftime('%b\n%y') for b in df['bulan']]

# ── 4A: Cadangan Klaim vs Total Nominal Klaim (dual axis) ───────────
ax4a = axes[0, 0]
ax4a_r = ax4a.twinx()

ax4a.fill_between(x, df['Cadangan Klaim']/1e6, alpha=0.2, color=C['cadangan'])
ax4a.plot(x, df['Cadangan Klaim']/1e6, color=C['cadangan'], lw=2.5,
          marker='o', ms=5, label='Cadangan Klaim (T Rp)')
ax4a_r.bar(x, df['total_nominal']/1e9, alpha=0.4, color=C['nominal'],
           width=0.6, label='Total Nominal Klaim Dibayar (Rp Miliar, kanan)')

ax4a.set_title('Cadangan Klaim vs Total Nominal Klaim Dibayar', fontweight='bold')
ax4a.set_ylabel('Cadangan Klaim (Triliun Rp)', color=C['cadangan'])
ax4a_r.set_ylabel('Total Nominal Dibayar (Rp Miliar)', color=C['nominal'])
ax4a.tick_params(axis='y', labelcolor=C['cadangan'])
ax4a.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:.0f}T'))
ax4a_r.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:.0f}M'))
ax4a.set_xticks(x); ax4a.set_xticklabels(xbl, fontsize=7)
lines1, lab1 = ax4a.get_legend_handles_labels()
lines2, lab2 = ax4a_r.get_legend_handles_labels()
ax4a.legend(lines1+lines2, lab1+lab2, loc='lower right', fontsize=7)

# ── 4B: Implied Reserve Ratio (berapa bulan beban yang di-cover) ────
ax4b = axes[0, 1]
if 'implied_reserve_months' in df.columns:
    irm = df['implied_reserve_months']
    colors_irm = [C['investasi'] if v >= 4 else (C['warn'] if v >= 2 else C['likuid'])
                  for v in irm.fillna(0)]
    bars4b = ax4b.bar(x, irm, color=colors_irm, alpha=0.85, edgecolor='white', width=0.7)
    ax4b.axhline(3, color='orange', lw=1.5, ls='--', label='3 bulan (buffer wajar)')
    ax4b.axhline(6, color='green', lw=1.2, ls=':', label='6 bulan (aman)')
    for bar, v in zip(bars4b, irm):
        if pd.notna(v):
            ax4b.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                      f'{v:.1f}x', ha='center', fontsize=7.5)
    ax4b.set_title('Implied Reserve Ratio\n= Cadangan Klaim / Nominal Klaim Bulan Ini\n'
                   '(berapa bulan beban yang ter-cover)', fontweight='bold')
    ax4b.set_ylabel('Bulan Coverage')
    ax4b.set_xticks(x); ax4b.set_xticklabels(xbl, fontsize=7)
    ax4b.legend(fontsize=8)

# ── 4C: Tren Kuartil Nominal (tail risk detection) ──────────────────
ax4c = axes[1, 0]
xq = range(len(quantile_monthly))
xqlabels = [b.strftime('%b\n%y') for b in quantile_monthly['bulan']]

ax4c.fill_between(xq, quantile_monthly['q25']/1e6, quantile_monthly['q75']/1e6,
                   alpha=0.2, color=C['klaim'], label='IQR (Q25-Q75)')
ax4c.plot(xq, quantile_monthly['q50']/1e6, color=C['klaim'], lw=2.2,
           marker='o', ms=4, label='Median')
ax4c.plot(xq, quantile_monthly['q95']/1e6, color=C['tail'], lw=2,
           ls='--', marker='s', ms=4, label='P95 (tail risk)')
ax4c.plot(xq, quantile_monthly['q99']/1e6, color=C['tail'], lw=1.5,
           ls=':', marker='^', ms=3, label='P99 (extreme tail)', alpha=0.7)

ax4c.set_title('Distribusi Nominal Klaim per Bulan\n(Tail Risk: apakah P95/P99 meningkat?)',
               fontweight='bold')
ax4c.set_ylabel('Nominal Klaim (Juta Rp)')
ax4c.set_xticks(xq); ax4c.set_xticklabels(xqlabels, fontsize=7)
ax4c.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'Rp{v:.0f}Jt'))
ax4c.legend(fontsize=8)

# ── 4D: Cadangan Premi vs Avg Rasio Coverage ─────────────────────────
ax4d = axes[1, 1]
if 'Cadangan Premi' in df.columns:
    ax4d_r = ax4d.twinx()
    ax4d.plot(x, df['Cadangan Premi']/1e6, color=C['investasi'], lw=2.2,
              marker='o', ms=5, label='Cadangan Premi (T Rp)')
    ax4d.fill_between(x, df['Cadangan Premi']/1e6, alpha=0.15, color=C['investasi'])
    ax4d_r.plot(x, df['avg_rasio_cov'], color=C['nominal'], lw=2, ls='--',
                marker='D', ms=4, label='Avg Rasio Coverage (kanan)')
    ax4d_r.axhline(1.0, color='red', lw=1, ls=':', alpha=0.6, label='Coverage = 100%')

    corr_premi_cov = df['Cadangan Premi'].corr(df['avg_rasio_cov'])
    ax4d.set_title(f'Cadangan Premi vs Rasio Coverage\n'
                   f'(r = {corr_premi_cov:.3f})', fontweight='bold')
    ax4d.set_ylabel('Cadangan Premi (Triliun Rp)', color=C['investasi'])
    ax4d_r.set_ylabel('Avg Rasio Coverage', color=C['nominal'])
    ax4d.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:.0f}T'))
    ax4d.set_xticks(x); ax4d.set_xticklabels(xbl, fontsize=7)
    lines1, lab1 = ax4d.get_legend_handles_labels()
    lines2, lab2 = ax4d_r.get_legend_handles_labels()
    ax4d.legend(lines1+lines2, lab1+lab2, fontsize=7)

plt.tight_layout()
plt.savefig('charts/analisis4_kecukupan_cadangan.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Insight Analisis 4 ──────────────────────────────────────────────
print('=== INSIGHT: KECUKUPAN CADANGAN ===')

if 'implied_reserve_months' in df.columns:
    irm = df['implied_reserve_months']
    print(f'\nImplied Reserve Months:')
    print(f'  Rata-rata : {irm.mean():.1f} bulan')
    print(f'  Terendah  : {irm.min():.1f} bulan ({df.loc[irm.idxmin(),"bulan"].strftime("%b-%Y")})')
    print(f'  Tertinggi : {irm.max():.1f} bulan ({df.loc[irm.idxmax(),"bulan"].strftime("%b-%Y")})')
    trend_irm = irm.iloc[-3:].mean() - irm.iloc[:3].mean()
    print(f'  Tren (rata3 awal vs akhir): {trend_irm:+.1f} bulan '
          f'({"MEMBAIK" if trend_irm > 0 else "MEMBURUK"})')

# Tail risk tren
q95_first3 = quantile_monthly['q95'].iloc[:3].mean()
q95_last3  = quantile_monthly['q95'].iloc[-3:].mean()
print(f'\nTail Risk (P95 nominal klaim):')
print(f'  3 bulan pertama: Rp{q95_first3/1e6:.0f}Jt')
print(f'  3 bulan terakhir: Rp{q95_last3/1e6:.0f}Jt')
print(f'  Perubahan: {(q95_last3-q95_first3)/q95_first3*100:+.1f}% '
      f'({"MENINGKAT ⚠" if q95_last3 > q95_first3 else "MENURUN ✓"})')

---
## Ringkasan Eksekutif

In [ ]:
print('=' * 65)
print('RINGKASAN EKSEKUTIF — EDA Klaim × Keuangan Perusahaan')
print('=' * 65)

print('\n[1] PROFIL RISIKO ICD')
print(f'    • Top beban: {icd_summary.iloc[0]["ICD_Group"]} '
      f'(Rp{icd_summary.iloc[0]["total_nom"]/1e9:.0f}M, '
      f'{icd_summary.iloc[0]["n_total"]} klaim)')
print(f'    • Penyakit paling mahal rata-rata: '
      f'{icd_summary.sort_values("avg_nom",ascending=False).iloc[0]["ICD_Group"]} '
      f'(avg Rp{icd_summary.sort_values("avg_nom",ascending=False).iloc[0]["avg_nom"]/1e6:.0f}Jt/klaim)')

print('\n[2] KLAIM LUAR NEGERI')
print(f'    • {n_ln/n_all*100:.1f}% klaim = LN, tapi {total_ln/total_all*100:.1f}% dari total nominal')
sg_mult = cost_by_loc.loc[cost_by_loc['lokasi_group']=='Singapore','multiplier']
if len(sg_mult):
    print(f'    • Singapore: {sg_mult.values[0]:.1f}x lebih mahal vs domestik')

print('\n[3] EFISIENSI PROSES')
print(f'    • Overall avg: {df_klaim["Processing_Days"].mean():.0f} hari')
print(f'    • Reimburse ({reimb_avg:.0f}h) vs Cashless ({cash_avg:.0f}h)')
print(f'    • LN ({ln_avg:.0f}h) vs Domestik ({dom_avg_p:.0f}h)')
if 'Utang Klaim' in df.columns:
    print(f'    • Korelasi proc_days vs Utang Klaim: {corr_proc_utang:.3f}')

print('\n[4] KECUKUPAN CADANGAN')
if 'implied_reserve_months' in df.columns:
    print(f'    • Implied Reserve avg: {df["implied_reserve_months"].mean():.1f} bulan beban')
print(f'    • P95 tail risk: {(q95_last3-q95_first3)/q95_first3*100:+.1f}% '
      f'(3 bulan pertama→terakhir)')
print('\n' + '=' * 65)